# Agent Saga：跨工具写操作的补偿与幂等

**面试问题：预订 Agent 跨库存、支付、物流三套系统写数据，部分失败时怎样回滚？**

## 回答主线

1. 分布式工具调用没有单数据库事务，不能假设三个写操作要么全成要么全败。
2. Saga 把业务拆成正向步骤及其补偿，失败后按已提交步骤的逆序执行补偿。
3. 补偿不是数据库 rollback，而是新的业务动作，因此也可能失败并需要重试。
4. 每个正向和补偿动作都必须携带稳定幂等键，重复回调不能二次扣款或二次退款。
5. 协调器要持久化意图和结果，解决副作用成功但本地日志尚未落盘的崩溃窗口。
6. 最终一致性评估必须查看库存、支付和物流三个权威状态。

## 真实案例

五个订单同时经过预留库存、扣款和创建物流：一个全成功，一个扣款失败，一个物流失败，一个收到重复回调，一个在副作用后崩溃。我们用内存服务复现状态和事件账本，重点观察补偿顺序与幂等键。数据是离线脱敏教学样本，只用于解释机制，不能宣称线上收益。

### 输入预览：五个订单及故障注入

In [1]:
orders = [  # 构造五个覆盖主要分布式故障的订单。
    {"id": "O1", "sku": "耳机", "amount": 399, "fail_at": None, "duplicate": False},  # 正常完成订单。
    {"id": "O2", "sku": "键盘", "amount": 599, "fail_at": "charge", "duplicate": False},  # 支付失败需要释放库存。
    {"id": "O3", "sku": "显示器", "amount": 1299, "fail_at": "shipment", "duplicate": False},  # 物流失败需要退款并释放库存。
    {"id": "O4", "sku": "鼠标", "amount": 199, "fail_at": None, "duplicate": True},  # 重复投递测试幂等。
    {"id": "O5", "sku": "硬盘", "amount": 699, "fail_at": "crash_after_charge", "duplicate": False},  # 扣款成功后协调器崩溃。
]  # 完成故障矩阵。
initial_inventory = {item["sku"]: 3 for item in orders}  # 为每个 SKU 准备三件教学库存。
print("订单  SKU    金额   故障点               重复投递")  # 输出 Saga 输入表头。
for order in orders:  # 逐订单展示故障注入配置。
    print(f"{order['id']}   {order['sku']:<5} {order['amount']:>4}  {str(order['fail_at']):<20} {order['duplicate']}")  # 展示业务与故障语义。

订单  SKU    金额   故障点               重复投递
O1   耳机     399  None                 False
O2   键盘     599  charge               False
O3   显示器   1299  shipment             False
O4   鼠标     199  None                 True
O5   硬盘     699  crash_after_charge   False


## Baseline 基线：无幂等键的正向调用与朴素重试

In [2]:
def naive_duplicate_charge(order):  # 模拟超时后直接重试扣款的错误基线。
    balance = 0  # 初始化商户已收金额。
    events = []  # 保存两次远端扣款事件。
    for attempt in range(2):  # 客户端因第一次响应丢失而重试两次。
        balance += order["amount"]  # 支付服务每次都产生真实副作用。
        events.append({"attempt": attempt + 1, "charged": order["amount"], "balance": balance})  # 记录重复扣款。
    return balance, events  # 返回错误余额和事件。

naive_balance, naive_events = naive_duplicate_charge(orders[3])  # 对重复投递订单运行基线。
print("无幂等重试账本：", naive_events)  # 展示同一订单被扣款两次。
print(f"应扣={orders[3]['amount']}，实际扣={naive_balance}，多扣={naive_balance - orders[3]['amount']}")  # 量化基线副作用。

无幂等重试账本： [{'attempt': 1, 'charged': 199, 'balance': 199}, {'attempt': 2, 'charged': 199, 'balance': 398}]
应扣=199，实际扣=398，多扣=199


### 核心实现：幂等服务与逆序补偿

In [3]:
def new_services():  # 创建隔离的库存、支付、物流和幂等状态。
    return {"inventory": initial_inventory.copy(), "reservations": set(), "charges": {}, "refunds": set(), "shipments": set(), "effects": {}, "ledger": []}  # 返回全新权威环境。

def apply_effect(services, key, operation, payload):  # 用幂等键包装所有外部副作用。
    if key in services["effects"]:  # 重复请求直接返回第一次结果。
        services["ledger"].append({"key": key, "operation": operation, "status": "deduplicated"})  # 记录去重事件。
        return services["effects"][key]  # 返回已持久化结果。
    if operation == "reserve":  # 处理库存预留。
        sku = payload["sku"]  # 读取目标 SKU。
        services["inventory"][sku] -= 1  # 扣减可售库存。
        services["reservations"].add(payload["order_id"])  # 标记订单已预留。
        result = {"ok": True}  # 返回预留成功。
    elif operation == "release":  # 处理库存补偿。
        sku = payload["sku"]  # 读取需要释放的 SKU。
        if payload["order_id"] in services["reservations"]:  # 只释放真实存在的预留。
            services["inventory"][sku] += 1  # 恢复可售库存。
            services["reservations"].remove(payload["order_id"])  # 删除预留状态。
        result = {"ok": True}  # 返回幂等释放成功。
    elif operation == "charge":  # 处理支付扣款。
        services["charges"][payload["order_id"]] = payload["amount"]  # 记录订单实际扣款。
        result = {"ok": True}  # 返回扣款成功。
    elif operation == "refund":  # 处理退款补偿。
        services["refunds"].add(payload["order_id"])  # 记录订单已退款。
        result = {"ok": True}  # 返回退款成功。
    elif operation == "shipment":  # 处理物流创建。
        services["shipments"].add(payload["order_id"])  # 记录已创建物流单。
        result = {"ok": True}  # 返回物流创建成功。
    else:  # 拒绝未知副作用。
        result = {"ok": False}  # 返回操作不存在。
    services["effects"][key] = result  # 在返回前持久化幂等结果。
    services["ledger"].append({"key": key, "operation": operation, "status": "applied"})  # 记录真实执行事件。
    return result  # 返回首次执行结果。

def run_saga(order, services):  # 执行三步预订 Saga 并在失败时逆序补偿。
    completed = []  # 保存已经提交的正向步骤。
    apply_effect(services, f"{order['id']}:reserve", "reserve", {"order_id": order["id"], "sku": order["sku"]})  # 预留库存。
    completed.append("reserve")  # 记录库存步骤已提交。
    if order["fail_at"] == "charge":  # 注入支付服务失败。
        failure = "charge-failed"  # 保存故障原因。
    else:  # 支付服务可执行。
        apply_effect(services, f"{order['id']}:charge", "charge", {"order_id": order["id"], "amount": order["amount"]})  # 使用稳定键扣款。
        completed.append("charge")  # 记录支付步骤已提交。
        failure = "coordinator-crash" if order["fail_at"] == "crash_after_charge" else None  # 模拟副作用后的协调器崩溃。
    if failure is None and order["fail_at"] == "shipment":  # 注入物流服务失败。
        failure = "shipment-failed"  # 保存物流故障。
    elif failure is None:  # 前两步和物流服务均可执行。
        apply_effect(services, f"{order['id']}:shipment", "shipment", {"order_id": order["id"]})  # 创建物流单。
        completed.append("shipment")  # 记录物流步骤已提交。
    if failure is not None and failure != "coordinator-crash":  # 可见业务失败立即进入补偿。
        for step in reversed(completed):  # 严格按正向提交的逆序补偿。
            if step == "charge":  # 支付步骤对应退款补偿。
                apply_effect(services, f"{order['id']}:refund", "refund", {"order_id": order["id"]})  # 发起一次幂等退款。
            if step == "reserve":  # 库存步骤对应释放补偿。
                apply_effect(services, f"{order['id']}:release", "release", {"order_id": order["id"], "sku": order["sku"]})  # 释放一次幂等库存。
    return {"order": order["id"], "completed": completed, "failure": failure}  # 返回协调器视角结果。

demo_services = new_services()  # 创建物流失败样本的隔离环境。
demo_result = run_saga(orders[2], demo_services)  # 执行 O3 并触发逆序补偿。
print("O3 Saga 结果：", demo_result)  # 展示已提交步骤与失败点。
print("O3 事件顺序：", [(event["operation"], event["status"]) for event in demo_services["ledger"]])  # 展示 reserve、charge、refund、release 顺序。

O3 Saga 结果： {'order': 'O3', 'completed': ['reserve', 'charge'], 'failure': 'shipment-failed'}
O3 事件顺序： [('reserve', 'applied'), ('charge', 'applied'), ('refund', 'applied'), ('release', 'applied')]


## 结果解读：五类终态逐系统核对

In [4]:
saga_rows = []  # 收集五个订单的终态评估。
for order in orders:  # 为每个故障样本创建独立服务状态。
    services = new_services()  # 隔离订单以便对比初始库存。
    result = run_saga(order, services)  # 执行当前 Saga。
    if order["duplicate"]:  # 对 O4 模拟整条消息重复投递。
        run_saga(order, services)  # 再次调用相同幂等键的正向步骤。
    saga_rows.append({"order": order["id"], "failure": result["failure"], "inventory": services["inventory"][order["sku"]], "charged": order["id"] in services["charges"], "refunded": order["id"] in services["refunds"], "shipment": order["id"] in services["shipments"], "charge_applied": sum(event["operation"] == "charge" and event["status"] == "applied" for event in services["ledger"]), "services": services})  # 保存跨系统终态。
print("订单  故障                库存  扣款  退款  物流  实际扣款次数")  # 输出终态表头。
for row in saga_rows:  # 逐订单展示最终一致性。
    print(f"{row['order']}   {str(row['failure']):<19} {row['inventory']:>4} {str(row['charged']):<5} {str(row['refunded']):<5} {str(row['shipment']):<5} {row['charge_applied']:>8}")  # 展示成功、补偿和去重结果。
print("解读：O2 仅释放库存；O3 先退款再释放；O4 重复投递只有一次真实扣款；O5 暴露副作用后崩溃窗口。")  # 解释不同终态。

订单  故障                库存  扣款  退款  物流  实际扣款次数
O1   None                   2 True  False True         1
O2   charge-failed          3 False False False        0
O3   shipment-failed        3 True  True  False        1
O4   None                   2 True  False True         1
O5   coordinator-crash      2 True  False False        1
解读：O2 仅释放库存；O3 先退款再释放；O4 重复投递只有一次真实扣款；O5 暴露副作用后崩溃窗口。


## 失败案例：扣款成功但协调器尚未记录完成就崩溃

In [5]:
crash_row = next(row for row in saga_rows if row["order"] == "O5")  # 读取副作用后崩溃样本。
crash_services = crash_row["services"]  # 获取支付服务的权威幂等记录。
recovered_charge = apply_effect(crash_services, "O5:charge", "charge", {"order_id": "O5", "amount": 699})  # 恢复器用同一键重放未知结果的扣款意图。
recovered_result = apply_effect(crash_services, "O5:shipment", "shipment", {"order_id": "O5"})  # 确认扣款存在后继续创建物流。
o5_charge_count = sum(event["operation"] == "charge" and event["status"] == "applied" for event in crash_services["ledger"])  # 统计真实扣款副作用次数。
print("恢复后最近事件：", crash_services["ledger"][-2:])  # 展示扣款被去重而物流首次应用。
print(f"恢复扣款结果={recovered_charge}，物流结果={recovered_result}，真实扣款次数={o5_charge_count}")  # 展示无二次扣款的恢复。
print("修正策略：先持久化带幂等键的步骤意图，再调用远端；恢复时查询或重放同一键，而不是生成新键。")  # 解释 Outbox/幂等恢复思想。

恢复后最近事件： [{'key': 'O5:charge', 'operation': 'charge', 'status': 'deduplicated'}, {'key': 'O5:shipment', 'operation': 'shipment', 'status': 'applied'}]
恢复扣款结果={'ok': True}，物流结果={'ok': True}，真实扣款次数=1
修正策略：先持久化带幂等键的步骤意图，再调用远端；恢复时查询或重放同一键，而不是生成新键。


### 生产边界与 Saga 状态

In [6]:
saga_contract = {"steps": ["reserve", "charge", "shipment"], "compensations": ["refund", "release"], "idempotency_scope": "order+operation", "recovery": "replay-same-key", "state_store": "durable-required"}  # 固化协调协议。
print("Saga 合同：", saga_contract)  # 展示必须版本化的步骤和恢复策略。
print("生产替换点：真实系统需要持久化状态机、Outbox/Inbox、补偿重试队列、人工悬挂处理、并发订单锁和财务对账。")  # 明确内存集合的边界。

Saga 合同： {'steps': ['reserve', 'charge', 'shipment'], 'compensations': ['refund', 'release'], 'idempotency_scope': 'order+operation', 'recovery': 'replay-same-key', 'state_store': 'durable-required'}
生产替换点：真实系统需要持久化状态机、Outbox/Inbox、补偿重试队列、人工悬挂处理、并发订单锁和财务对账。


## 回归测试：最后只保护逆序补偿、幂等与崩溃恢复

In [7]:
assert [event["operation"] for event in demo_services["ledger"]] == ["reserve", "charge", "refund", "release"]  # 验证物流失败后按逆序补偿。
assert saga_rows[1]["inventory"] == 3 and not saga_rows[1]["charged"]  # 验证支付失败只释放库存。
assert saga_rows[2]["inventory"] == 3 and saga_rows[2]["refunded"]  # 验证物流失败恢复库存并退款。
assert saga_rows[3]["charge_applied"] == 1  # 验证整条消息重复投递不会二次扣款。
assert o5_charge_count == 1 and "O5" in crash_services["shipments"]  # 验证崩溃恢复去重旧扣款并继续物流。
print("回归测试通过：补偿逆序、支付失败、物流失败、重复投递和副作用后崩溃恢复均成立。")  # 用少量断言总结 Saga 合同。

回归测试通过：补偿逆序、支付失败、物流失败、重复投递和副作用后崩溃恢复均成立。
